# SNR equalisation

The LOS clusters have an SNR of about 38 dB and the NLOS clusters about 19 dB. This notebook adds complex Gaussian noise that follows the measured noise floor to the LOS clusters until their SNR equals the mean NLOS SNR, and evaluates all four models on the 80 splits. Condition A (`equalised`) trains and tests on the equalised data. Condition B (`test_shift`) trains on the original data and tests on the equalised LOS clusters.

In [ ]:
import os
import sys
import json
import time
import importlib
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import glob
import shutil

def find_code_dir():
    here = os.path.abspath("..")
    if os.path.exists(os.path.join(here, "common.py")):
        return here
    hits = glob.glob("/kaggle/input/**/robustness/common.py", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError("robustness/common.py not found (locally or under /kaggle/input)")

CODE_DIR = find_code_dir()
sys.path.insert(0, CODE_DIR)
import common as cm
importlib.reload(cm)

RESULTS_DIR = os.path.abspath("results")
FRAME_MODELS = ["svm", "rf"]
DEEP_MODELS = ["cnn_transformer", "cnn_lstm"]
MODES = ["equalised", "test_shift"]
PROTOCOLS = ["lopo", "l2po"]
N_PARALLEL = 2
TARGET_SNR_DB = None
NOISE_SEED = 0
REBUILD_DATA = False
RESUME_FROM = None

def resume(results_dir, source):
    if not source:
        return
    for f in glob.glob(os.path.join(source, "*.jsonl")):
        dst = os.path.join(results_dir, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copyfile(f, dst)
    if os.path.isdir(os.path.join(source, "frame_scores")):
        shutil.copytree(os.path.join(source, "frame_scores"), os.path.join(results_dir, "frame_scores"),
                        dirs_exist_ok=True)
    print("resumed from", source)

os.makedirs(RESULTS_DIR, exist_ok=True)
resume(RESULTS_DIR, RESUME_FROM)
print("Python", sys.version.split()[0], "|", sys.executable)
print("original data :", cm.ORIG_DATA_DIR)
print("SNR-equalised :", cm.SNR_DATA_DIR)
print("raw traces    :", cm.RAW_DIR)
HAVE_RAW = os.path.isdir(cm.RAW_DIR) and all(os.path.exists(cm.csi_path(c)) for c in cm.ALL_CLUSTERS)
print("raw traces and CSI available:", HAVE_RAW, "(not needed if the SNR-equalised dataset was built locally)")

## Step 1: SNR of every cluster

Signal power from the calibrated CSI and noise power from the noise-only slot, per antenna and subcarrier. Needs the raw traces.

In [ ]:
AUDIT_SNR = {"LOS_cluster1": 39.2, "LOS_cluster2": 38.9, "LOS_cluster3": 37.1, "LOS_cluster4": 38.6,
             "NLOS_cluster1": 18.8, "NLOS_cluster2": 17.9, "NLOS_cluster3": 19.1, "NLOS_cluster4": 18.6,
             "NLOS_cluster5": 19.0}
if not HAVE_RAW:
    snr = None
    print("Raw traces / CSI are not available here (e.g. on Kaggle): Steps 1-2 are skipped.")
    print("The SNR values are read in Step 3 from the dataset built locally (index.json).")
else:
    snr = cm.measure_snr()
print()
print(f"{'cluster':14s} {'measured':>9s} {'reference':>9s}")
for c in cm.ALL_CLUSTERS if snr else []:
    print(f"{c:14s} {snr[c]['snr_db']:8.1f}  {AUDIT_SNR[c]:8.1f}")

## Step 2: Reproduction check

Features recomputed from the stored CSI without noise must equal the pipeline features.

In [ ]:
for c in cm.LOS_CLUSTERS if HAVE_RAW else []:
    d = cm.check_reproduction(c)
    print(f"{c:14s} max |difference| = {d:.2e}")
    assert d < 1e-4, "features could not be reproduced from the CSI"
print("OK: features are reproduced exactly" if HAVE_RAW else "skipped (no CSI here; checked locally)")

## Step 3: SNR-equalised dataset

Written to `../data/snr_equalised/clusters/` with the same layout as `output_raw_full/clusters/`.

In [ ]:
idx_path = os.path.join(cm.SNR_DATA_DIR, "clusters", "index.json")
if os.path.exists(idx_path) and not REBUILD_DATA:
    with open(idx_path) as f:
        info = json.load(f)["snr_equalisation"]
    print("using the existing SNR-equalised dataset (set REBUILD_DATA = True to rebuild)")
elif not HAVE_RAW:
    raise FileNotFoundError("No SNR-equalised dataset found and no raw traces to build it: run "
                            "make_kaggle_bundle.py locally first.")
else:
    t0 = time.time()
    info = cm.build_snr_equalised_dataset(snr, TARGET_SNR_DB, seed=NOISE_SEED)
    print(f"built in {time.time() - t0:.0f} s")

print(f"target SNR: {info['target_snr_db']:.1f} dB")
print(f"{'cluster':14s} {'SNR before':>10s} {'SNR after':>10s} {'noise x':>8s} {'CV before':>10s} {'CV after':>9s}")
for c in cm.ALL_CLUSTERS:
    e = info["clusters"][c]
    before = e.get("snr_before_db", e.get("snr_db"))
    after = e.get("snr_after_db", e.get("snr_db"))
    g = e.get("noise_factor_g", 1.0)
    print(f"{c:14s} {before:9.1f}  {after:9.1f}  {g:7.1f}  "
          f"{cm.frequency_cv(cm.ORIG_DATA_DIR, c):9.3f}  {cm.frequency_cv(cm.SNR_DATA_DIR, c):8.3f}")

## Step 4: SVM and RF

In [ ]:
X_orig, y = cm.load_frame_features(cm.ORIG_DATA_DIR)
X_snr, _ = cm.load_frame_features(cm.SNR_DATA_DIR)

for model in FRAME_MODELS:
    for mode in MODES:
        for protocol in PROTOCOLS:
            path = os.path.join(RESULTS_DIR, f"frame_{model}_{mode}_{protocol}.jsonl")
            done = cm.load_done(path)
            splits = cm.make_splits(protocol)
            print(f"=== {model} / {mode} / {cm.PROTOCOLS[protocol]}: {len(splits)} splits ({len(done)} already done)")
            t_start = time.time()
            for i, sp in enumerate(splits, 1):
                if sp["name"] in done:
                    continue
                if mode == "equalised":
                    r = cm.evaluate_frame_split(model, sp, X_snr, y, {"test": X_snr})
                    msg = f"acc={r['test']['frame']['accuracy']:.4f}"
                else:
                    r = cm.evaluate_frame_split(model, sp, X_orig, y, {"original": X_orig, "shifted": X_snr})
                    msg = (f"original={r['original']['frame']['accuracy']:.4f}  "
                           f"shifted={r['shifted']['frame']['accuracy']:.4f}")
                cm.append_result(path, r)
                print(f"  [{i:2d}/{len(splits)}] {sp['name']:32s} {msg}  ({time.time() - t_start:.0f}s)")

## Step 5: CNN-Transformer and CNN-LSTM

Each split runs in its own process (`../run_deep_split.py`) with the stored configuration `results/part_b_config.json` of each model.

In [ ]:
WORKER = os.path.join(CODE_DIR, "run_deep_split.py")
lock = threading.Lock()

def run_deep(model, mode, split):
    arg = json.dumps({k: split[k] for k in ("name", "train", "test")})
    env = dict(os.environ, OMP_NUM_THREADS=str(max(1, (os.cpu_count() or 2) // N_PARALLEL)),
               TF_FORCE_GPU_ALLOW_GROWTH="true")
    proc = subprocess.run([sys.executable, WORKER, "--model", model, "--mode", mode, "--split", arg],
                          capture_output=True, text=True, env=env)
    lines = [l for l in proc.stdout.splitlines() if l.startswith("RESULT ")]
    if proc.returncode != 0 or not lines:
        raise RuntimeError(f"{model} / {mode} / {split['name']} failed (exit {proc.returncode}):\n{proc.stderr[-3000:]}")
    return json.loads(lines[-1][len("RESULT "):])

for mode in MODES:
    for model in DEEP_MODELS:
        for protocol in PROTOCOLS:
            path = os.path.join(RESULTS_DIR, f"deep_{model}_{mode}_{protocol}.jsonl")
            done = cm.load_done(path)
            splits = cm.make_splits(protocol)
            todo = [sp for sp in splits if sp["name"] not in done]
            print(f"=== {model} / {mode} / {cm.PROTOCOLS[protocol]}: {len(splits)} splits ({len(done)} already done)")
            t_start = time.time()
            with ThreadPoolExecutor(max_workers=N_PARALLEL) as pool:
                futures = {pool.submit(run_deep, model, mode, sp): sp for sp in todo}
                for k, fut in enumerate(as_completed(futures), 1):
                    r = fut.result()
                    with lock:
                        cm.append_result(path, r)
                    extra = f"  shifted={r['shifted_test']['accuracy']:.4f}" if mode == "test_shift" else ""
                    print(f"  [{len(done) + k:2d}/{len(splits)}] {r['name']:32s} acc={r['accuracy']:.4f}{extra}  "
                          f"epochs={r['selected_epochs']}  ({time.time() - t_start:.0f}s)")

## Step 6: Comparison with the stored results

Mean ± standard deviation over splits, LOS and NLOS recall and the paired Wilcoxon test against the stored results.

In [ ]:
STORED = {"svm": "SVM", "rf": "RF", "cnn_lstm": "CNN_LSTM", "cnn_transformer": "CNN_Transformer"}
NAMES = {"svm": "SVM", "rf": "RF", "cnn_lstm": "CNN-LSTM", "cnn_transformer": "CNN-Transformer"}

def stored(model, protocol):
    return cm.load_done(os.path.join(cm.PV2, STORED[model], "results", f"splits_{protocol}.jsonl"))

def new(model, mode, protocol):
    kind = "frame" if model in ("svm", "rf") else "deep"
    return cm.load_done(os.path.join(RESULTS_DIR, f"{kind}_{model}_{mode}_{protocol}.jsonl"))

def view(model, mode, r):
    if model in ("svm", "rf"):
        return r["test"]["frame"] if mode == "equalised" else r["shifted"]["frame"]
    return r if mode == "equalised" else r["shifted_test"]

def fmt(a):
    return f"{100 * np.mean(a):6.2f} ± {100 * np.std(a):5.2f}" if len(a) else "      --      "

rows, summary = [], {}
for model in FRAME_MODELS + DEEP_MODELS:
    for protocol in PROTOCOLS:
        base = stored(model, protocol)
        line = {"model": NAMES[model], "protocol": cm.PROTOCOLS[protocol]}
        line["stored"] = [base[n]["accuracy"] for n in base]
        for mode in ("equalised", "test_shift"):
            res = new(model, mode, protocol)
            names = [n for n in base if n in res]
            line[mode] = {"names": names,
                          "acc": [view(model, mode, res[n])["accuracy"] for n in names],
                          "rec_los": [view(model, mode, res[n])["recall_los"] for n in names],
                          "rec_nlos": [view(model, mode, res[n])["recall_nlos"] for n in names],
                          "base": [base[n]["accuracy"] for n in names]}
            if mode == "test_shift" and names:
                orig = [(res[n]["original"]["frame"] if model in ("svm", "rf") else res[n])["accuracy"] for n in names]
                line["sanity_max_diff"] = float(np.max(np.abs(np.array(orig) - np.array(line[mode]["base"]))))
        tsr = new(model, "test_shift", protocol)
        rn = [n for n in line["equalised"]["names"] if n in tsr]
        line["rerun"] = {"names": rn,
                         "acc": [(tsr[n]["original"]["frame"] if model in ("svm", "rf") else tsr[n])["accuracy"] for n in rn],
                         "eq": [view(model, "equalised", new(model, "equalised", protocol)[n])["accuracy"] for n in rn]}
        if len(rn) >= 6:
            d = np.array(line["rerun"]["eq"]) - np.array(line["rerun"]["acc"])
            line["p_vs_rerun"] = float(stats.wilcoxon(line["rerun"]["eq"], line["rerun"]["acc"]).pvalue) if np.any(d != 0) else 1.0
        a = line["equalised"]
        if len(a["names"]) >= 6:
            d = np.array(a["acc"]) - np.array(a["base"])
            line["p_equalised"] = float(stats.wilcoxon(a["acc"], a["base"]).pvalue) if np.any(d != 0) else 1.0
        rows.append(line)

print(f"{'model':16s} {'prot':5s} {'stored (%)':>15s} {'A: equalised (%)':>17s} {'LOS/NLOS rec. A':>16s} "
      f"{'p (A vs stored)':>15s} {'B: test shift (%)':>18s} {'LOS/NLOS rec. B':>16s} {'sanity':>8s}")
for l in rows:
    A, B = l["equalised"], l["test_shift"]
    recA = f"{100*np.mean(A['rec_los']):5.1f}/{100*np.mean(A['rec_nlos']):5.1f}" if A["acc"] else "--"
    recB = f"{100*np.mean(B['rec_los']):5.1f}/{100*np.mean(B['rec_nlos']):5.1f}" if B["acc"] else "--"
    p = f"{l['p_equalised']:.2g}" if "p_equalised" in l else "--"
    s = f"{l['sanity_max_diff']:.3f}" if "sanity_max_diff" in l else "--"
    print(f"{l['model']:16s} {l['protocol']:5s} {fmt(l['stored'])} {fmt(A['acc'])} {recA:>16s} {p:>15s} "
          f"{fmt(B['acc'])}   {recB:>16s} {s:>8s}")
    print(f"{'':22s} splits done: A {len(A['acc'])}, B {len(B['acc'])}")
    if l["rerun"]["acc"]:
        pr = f"{l['p_vs_rerun']:.2g}" if "p_vs_rerun" in l else "--"
        print(f"{'':22s} same machine: re-run original {fmt(l['rerun']['acc'])}  vs  A {fmt(l['rerun']['eq'])}  (p = {pr})")

with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump({"snr": {c: info["clusters"][c] for c in cm.ALL_CLUSTERS}, "target_snr_db": info["target_snr_db"],
               "rows": rows}, f, indent=1)
print("\nwritten:", os.path.join(RESULTS_DIR, "summary.json"))

## Step 7: Figure

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharex=True, sharey=True)
for ax, model in zip(axes, FRAME_MODELS + DEEP_MODELS):
    for protocol, mk in (("lopo", "o"), ("l2po", "s")):
        l = next(r for r in rows if r["model"] == NAMES[model] and r["protocol"] == cm.PROTOCOLS[protocol])
        A = l["equalised"]
        if A["acc"]:
            ax.scatter(100 * np.array(A["base"]), 100 * np.array(A["acc"]), s=16, marker=mk, alpha=0.65,
                       color="#2a78d6" if model.startswith("cnn") else "#8a8984", label=cm.PROTOCOLS[protocol])
    ax.plot([0, 100], [0, 100], color="#8a8984", lw=0.8, ls="--")
    ax.set_xlim(20, 101); ax.set_ylim(20, 101)
    ax.set_title(NAMES[model]); ax.set_xlabel("stored accuracy (%)")
    ax.grid(color="#e6e5e0", lw=0.6)
axes[0].set_ylabel("accuracy with equalised SNR (%)")
axes[0].legend(frameon=False, loc="lower right")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(RESULTS_DIR, f"snr_equalisation.{ext}"), dpi=200, bbox_inches="tight")
plt.show()